## Multicollinearity

### Problem: Predicting Salary with Correlated Features

- Imagine we want to predict a person’s salary based on education and years of training in skills.

  - Education level (1–5)
  - Training hours (highly correlated with education)
  - Projects completed

We want to see how multicollinearity affects coefficients.

### Important: 

1. **Consider this → We want to predict salary using**:
   - Education
   - Training hours
----

2. **Now suppose → Every person with more education ALSO always has more training hours.**
----

3. **Example:**

| Education level| Training Hours |
| -----------    | -------------- |
| 1         | 20             |
| 2         | 40             |
| 3         | 60             |
| 4         | 80             |


 **Now here: Training hours = just another version of education**

----

4. **Now the problem**
  - The Model is trying to learn:
    - “How much does education affect salary?”
    - “How much does training affect salary?”


BUT, both are telling the same story. So the model gets confused:

> “It wonders, if salary increases… is it because of education OR training??”

`That confusion = multicollinearity

### Step 1: Creating synthetic data

In below code, we intentionally created multicollinearity.

 - training_hours = education * 20 + noise

 - Training_hours ≈ Education → SAME information

In [9]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Feature 1
education = np.random.randint(1, 6, 100)

# Feature 2 (collinear with education)
training_hours = education * 20 + np.random.normal(0, 5, 100)

# Independent feature
projects = np.random.randint(0, 20, 100)

# Target
salary = (
    500 * education +
    10 * training_hours +
    200 * projects +
    np.random.normal(0, 50, 100)
)

# DataFrame
df = pd.DataFrame({
    "Education": education,
    "Training_Hours": training_hours,
    "Projects": projects,
    "Salary": salary
})

df.head()

,Education,Training_Hours,Projects,Salary
0,4,75.518324,18,6277.155387
1,5,99.440061,15,6594.418013
2,3,67.344706,15,5113.440477
3,5,94.380508,2,3903.874105
4,5,104.750027,19,7354.991956


### Step 2: Check Correlation

#### Correlation basics
- Correlation ranges from -1 to +1
  1. +1 → perfect positive relationship
  2. 0 → no relationship
  3. -1 → perfect negative relationship


   
- Values close to ±1 → strong relationship



In [3]:
df.corr()

,Education,Training_Hours,Projects,Salary
Education,1.000000,0.983940,0.108674,0.661844
Training_Hours,0.983940,1.000000,0.117945,0.664404
Projects,0.108674,0.117945,1.000000,0.815867
Salary,0.661844,0.664404,0.815867,1.000000


**Key observations**

1. Education & Training_Hours = 0.983940
  - Very high correlation → almost duplicate information
This is multicollinearity.

2. Projects
  - Correlation with Education = 0.108674 → low
  - Correlation with Training_Hours = 0.117945 → low
  - Projects is independent → good for regression
    
3. Salary correlations
  - Salary vs Projects = 0.815867 → strong positive effect
  - Salary vs Education = 0.661844 → moderate
  - Salary vs Training_Hours = 0.664404 → moderate

**What this tells us?**

- Multicollinearity exists between Education & Training_Hours
- The model might have unstable coefficients for these two features
- **Predictions will still be okay**
- Interpretation of how much Education vs Training affects Salary → tricky

### Step 3: Train Model (with multicollinearity)

In [10]:
from sklearn.linear_model import LinearRegression

X = df[["Education", "Training_Hours", "Projects"]]
y = df["Salary"]

model = LinearRegression()
model.fit(X, y)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

Coefficients: [516.36180537   9.25511232 198.69834332]
Intercept: 7.0645414727532625
